In [ ]:
from pathlib import Path
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Optional
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from feature_engine.datetime import DatetimeFeatures
from feature_engine.transformation import LogTransformer
from feature_engine.imputation import MeanMedianImputer
from feature_engine.encoding import RareLabelEncoder

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'data' / 'processed' / 'flowcred_cleaned.csv').exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CLEAN_PATH = REPO_ROOT / 'data' / 'processed' / 'flowcred_cleaned.csv'
FEATURES_PATH = REPO_ROOT / 'data' / 'processed' / 'payment_delay_features.csv'
TRAIN_PATH = REPO_ROOT / 'data' / 'processed' / 'payment_delay_train.csv'
VAL_PATH = REPO_ROOT / 'data' / 'processed' / 'payment_delay_validation.csv'
TEST_PATH = REPO_ROOT / 'data' / 'processed' / 'payment_delay_test.csv'
PIPE_PATH = REPO_ROOT / 'models' / 'payment_delay_feature_pipeline.joblib'
DICT_PATH = REPO_ROOT / 'reports' / 'payment_delay_feature_dictionary.csv'
AUDIT_PATH = REPO_ROOT / 'reports' / 'payment_delay_leakage_audit.csv'
DQ_PATH = REPO_ROOT / 'reports' / 'payment_delay_data_quality_report.csv'
PLOT_DIR = REPO_ROOT / 'reports' / 'payment_delay'
PLOT_DIR.mkdir(parents=True, exist_ok=True)
TARGET = 'delay_days'
RANDOM_STATE = 42

print('REPO_ROOT:', REPO_ROOT)
print('INPUT:', CLEAN_PATH)
print('PLOT_DIR:', PLOT_DIR)

## 1. Project Objective

Predict the signed continuous target `delay_days = payment_date - due_date`:

- `delay_days < 0` -> paid early; `= 0` -> on time; `> 0` -> paid late.
- Keep negatives as-is (do NOT clip to zero).
- Supervised training uses ONLY paid invoices (`payment_date` not null, valid `due_date`/`delay_days`); open invoices have unknown delay and are excluded.
- Feature-engine is the primary framework (`DatetimeFeatures`, `LogTransformer`, `MeanMedianImputer`, `RareLabelEncoder`); customer history uses a custom sklearn-compatible `LeakageSafeCustomerHistoryTransformer` so temporal correctness is never compromised.
- Chronological 70/15/15 split by (`invoice_date`, `invoice_id`); all learned transforms fit on TRAIN only.

## 2. Imports and Configuration

All imports, paths, constants and plot directory are set up in the cell below (executed first). No model classes are imported — this notebook never trains regressors.

In [ ]:
import sklearn
import feature_engine
print('sklearn:', sklearn.__version__, '| feature_engine:', feature_engine.__version__)
assert CLEAN_PATH.exists(), f'Input not found: {CLEAN_PATH}'
FORBIDDEN = ['payment_date', 'delay_days', 'is_late', 'invoice_status', 'invoice_id', 'customer_id']
print('Forbidden-as-feature columns:', FORBIDDEN)

## 3. Load Cleaned Dataset

Load `data/processed/flowcred_cleaned.csv` without modification.

In [ ]:
df_raw = pd.read_csv(CLEAN_PATH)
print('shape:', df_raw.shape)
print('columns:', list(df_raw.columns))
print(df_raw.head(3).to_string())
assert len(df_raw) > 0, 'Empty input dataset'

## 4. Dataset Audit

Audit schema, missing values, duplicates, statuses and candidate categoricals before any transformation.

In [ ]:
def audit_frame(df: pd.DataFrame, name: str) -> None:
    """Print shape, dtypes, missing counts, duplicates for a dataframe."""
    print(f'--- {name}: shape {df.shape} ---')
    print(df.dtypes.to_string())
    miss = df.isna().sum()
    print('missing per column:')
    print(miss[miss > 0].to_string() if (miss > 0).any() else '  none')
    print('duplicated rows:', int(df.duplicated().sum()))

audit_frame(df_raw, 'flowcred_cleaned')
print('invoice_status counts:')
print(df_raw['invoice_status'].value_counts(dropna=False).to_string())
print('cust_payment_terms top 10:')
print(df_raw['cust_payment_terms'].value_counts(dropna=False).head(10).to_string())
print('invoice_currency counts:')
print(df_raw['invoice_currency'].value_counts(dropna=False).to_string())
assert df_raw['customer_id'].isna().sum() == 0, 'Missing customer_id needs handling (none expected)'
assert df_raw['invoice_amount'].isna().sum() == 0, 'Missing invoice_amount found'
assert (df_raw['invoice_amount'] <= 0).sum() == 0, 'Non-positive amounts found (log unsafe)'

## 5. Date Conversion

Convert `invoice_date`, `due_date`, `payment_date` to datetime; report invalid dates and impossible relationships (`due < invoice`).

In [ ]:
df = df_raw.copy()
for col in ['invoice_date', 'due_date', 'payment_date']:
    df[col] = pd.to_datetime(df[col], errors='coerce')
print('invalid invoice_date:', int(df['invoice_date'].isna().sum()))
print('invalid due_date:', int(df['due_date'].isna().sum()))
print('invalid payment_date (incl. open):', int(df['payment_date'].isna().sum()))
bad_due = int(((df['due_date'].notna()) & (df['invoice_date'].notna()) & (df['due_date'] < df['invoice_date'])).sum())
print('rows with due_date < invoice_date (flagged, kept):', bad_due)
assert df['invoice_date'].isna().sum() == 0, 'Unparseable invoice_date found'
assert df['due_date'].isna().sum() == 0, 'Unparseable due_date found'

## 6. Create delay_days Target

Recompute `delay_days = (payment_date - due_date).dt.days` (signed), compare with the stored column, and validate the distribution. Negatives are kept.

In [ ]:
df['delay_days_calc'] = (df['payment_date'] - df['due_date']).dt.days
stored = df['delay_days']
calc = df['delay_days_calc']
mismatch = int((((stored.isna()) != (calc.isna())) | (stored.fillna(-99999) != calc.fillna(-99999))).sum())
print('mismatches between stored delay_days and recomputed:', mismatch)
assert mismatch == 0, 'Stored delay_days disagrees with payment_date - due_date'
df[TARGET] = calc  # canonical signed continuous target
paid_mask = df['payment_date'].notna()
print('delay_days describe (paid only):')
print(df.loc[paid_mask, TARGET].describe().to_string())
print('negative delays:', int((df.loc[paid_mask, TARGET] < 0).sum()))
print('zero delays:', int((df.loc[paid_mask, TARGET] == 0).sum()))
print('positive delays:', int((df.loc[paid_mask, TARGET] > 0).sum()))
print('extreme delays (|delay|>180):', int((df.loc[paid_mask, TARGET].abs() > 180).sum()), '(flagged, NOT silently removed)')
assert (df.loc[paid_mask, TARGET] < 0).sum() > 0, 'Expected some early payments (negatives)'

## 7. Filter Paid Invoices

Keep ONLY rows with non-null `payment_date`, valid `due_date` and valid `delay_days`. Open invoices are excluded from supervised regression (unknown target). A data-quality report is built and saved later.

In [ ]:
total_rows = len(df)
n_open = int(df['payment_date'].isna().sum())
paid_df = df[df['payment_date'].notna() & df['due_date'].notna() & df[TARGET].notna()].copy()
print('total rows:', total_rows)
print('paid (usable):', len(paid_df))
print('open/excluded:', total_rows - len(paid_df))
assert len(paid_df) == int((df_raw['invoice_status'] == 'paid').sum()), 'Paid filter disagrees with invoice_status'
assert paid_df['payment_date'].isna().sum() == 0
assert paid_df[TARGET].isna().sum() == 0
dq_report = pd.DataFrame([
    {'metric': 'total_rows', 'value': total_rows},
    {'metric': 'paid_invoices', 'value': len(paid_df)},
    {'metric': 'unpaid_open_invoices', 'value': total_rows - len(paid_df)},
    {'metric': 'missing_payment_date', 'value': int(df['payment_date'].isna().sum())},
    {'metric': 'missing_due_date', 'value': int(df['due_date'].isna().sum())},
    {'metric': 'missing_delay_days', 'value': int(df[TARGET].isna().sum() - n_open)},
    {'metric': 'negative_delay_days', 'value': int((paid_df[TARGET] < 0).sum())},
    {'metric': 'zero_delay_days', 'value': int((paid_df[TARGET] == 0).sum())},
    {'metric': 'positive_delay_days', 'value': int((paid_df[TARGET] > 0).sum())},
    {'metric': 'n_customers_paid', 'value': int(paid_df['customer_id'].nunique())},
])
print(dq_report.to_string(index=False))

## 8. Chronological Sorting

Sort by `invoice_date` then `invoice_id` (deterministic tie-breaker) and assert monotonicity. Same-day invoices are processed as a batch later so they never see each other as history.

In [ ]:
paid_df = paid_df.sort_values(['invoice_date', 'invoice_id'], kind='mergesort').reset_index(drop=True)
assert paid_df['invoice_date'].is_monotonic_increasing, 'Paid data not chronologically sorted'
print('sorted range:', paid_df['invoice_date'].min(), '->', paid_df['invoice_date'].max())
print(paid_df[['invoice_id', 'customer_id', 'invoice_date', 'due_date', TARGET]].head(3).to_string(index=False))

## 9. Leakage-Safe Customer History Feature Engineering

Custom `LeakageSafeCustomerHistoryTransformer` (sklearn `BaseEstimator`/`TransformerMixin`): per customer, in chronological order, each same-day batch receives aggregates over **strictly earlier** batches only (A->none; B->A; C->A+B; ...). Late means previous `delay_days > 0`. No-history defaults: counts `0`, `previous_late_ratio` documented neutral `0.0`, delay/amount averages `NaN` (imputed later), totals `0`, recency `NaN`, `has_payment_history 0`. History is computed on the FULL paid table BEFORE the split so validation/test invoices keep valid earlier (incl. train-period) history — never future data.

In [ ]:
class LeakageSafeCustomerHistoryTransformer(BaseEstimator, TransformerMixin):
    """Previous-only customer aggregates for payment-delay regression.

    Late = previous delay_days > 0. Same-day batches never see each other.
    No-history defaults: counts 0, late_ratio 0.0 (neutral), delay/amount
    means NaN (imputed later), totals 0, recency NaN, has_payment_history 0.
    """
    def __init__(self, customer_col: str = 'customer_id', date_col: str = 'invoice_date',
                 delay_col: str = 'delay_days', amount_col: str = 'invoice_amount',
                 invoice_id_col: str = 'invoice_id') -> None:
        self.customer_col = customer_col
        self.date_col = date_col
        self.delay_col = delay_col
        self.amount_col = amount_col
        self.invoice_id_col = invoice_id_col

    def fit(self, X: pd.DataFrame, y=None):
        X = self._as_df(X)
        missing = [c for c in [self.customer_col, self.date_col, self.delay_col, self.amount_col] if c not in X.columns]
        if missing:
            raise ValueError(f'Missing columns for history: {missing}')
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = self._as_df(X)
        w = X.copy()
        w['_hdate'] = pd.to_datetime(w[self.date_col], errors='coerce')
        w['_delay'] = pd.to_numeric(w[self.delay_col], errors='coerce')
        w['_amt'] = pd.to_numeric(w[self.amount_col], errors='coerce')
        keys = [self.customer_col, '_hdate'] + ([self.invoice_id_col] if self.invoice_id_col in w.columns else [])
        w = w.sort_values(keys, kind='mergesort')
        cols = ['previous_invoice_count', 'previous_paid_invoice_count', 'previous_late_count',
                'previous_late_ratio', 'previous_avg_delay', 'previous_median_delay',
                'previous_max_delay', 'previous_min_delay', 'previous_avg_invoice_amount',
                'previous_total_invoice_amount', 'previous_days_since_invoice', 'has_payment_history']
        out = pd.DataFrame(index=w.index, columns=cols)
        out['previous_invoice_count'] = 0
        out['previous_paid_invoice_count'] = 0
        out['previous_late_count'] = 0
        out['previous_late_ratio'] = 0.0
        out[['previous_avg_delay', 'previous_median_delay', 'previous_max_delay', 'previous_min_delay',
             'previous_avg_invoice_amount']] = np.nan
        out['previous_total_invoice_amount'] = 0.0
        out['previous_days_since_invoice'] = np.nan
        out['has_payment_history'] = 0
        for _, g in w.groupby(self.customer_col, sort=False):
            prior_delays: List[float] = []
            prior_amts: List[float] = []
            last_date: Optional[pd.Timestamp] = None
            dated = g[g['_hdate'].notna()]
            for cur_date, batch in dated.groupby('_hdate', sort=True):
                idx = batch.index
                n = len(prior_delays)
                if n:
                    late = [1 for d in prior_delays if d > 0]
                    out.loc[idx, 'previous_invoice_count'] = n
                    out.loc[idx, 'previous_paid_invoice_count'] = n  # supervised frame is paid-only
                    out.loc[idx, 'previous_late_count'] = int(sum(late))
                    out.loc[idx, 'previous_late_ratio'] = float(sum(late) / n)
                    out.loc[idx, 'previous_avg_delay'] = float(np.mean(prior_delays))
                    out.loc[idx, 'previous_median_delay'] = float(np.median(prior_delays))
                    out.loc[idx, 'previous_max_delay'] = float(np.max(prior_delays))
                    out.loc[idx, 'previous_min_delay'] = float(np.min(prior_delays))
                    if prior_amts:
                        out.loc[idx, 'previous_avg_invoice_amount'] = float(np.mean(prior_amts))
                        out.loc[idx, 'previous_total_invoice_amount'] = float(sum(prior_amts))
                    out.loc[idx, 'previous_days_since_invoice'] = float((cur_date - last_date).days)
                    out.loc[idx, 'has_payment_history'] = 1
                prior_delays.extend(batch['_delay'].dropna().tolist())
                prior_amts.extend(batch['_amt'].dropna().tolist())
                last_date = cur_date
        return out.reindex(X.index).astype({
            'previous_invoice_count': 'int64', 'previous_paid_invoice_count': 'int64',
            'previous_late_count': 'int64', 'has_payment_history': 'int64'})

    @staticmethod
    def _as_df(X) -> pd.DataFrame:
        return X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)

hist_tx = LeakageSafeCustomerHistoryTransformer()
hist_tx.fit(paid_df)
hist_feats = hist_tx.transform(paid_df)
print(hist_feats.shape)
print(hist_feats.head(3).to_string())
assert (hist_feats.loc[hist_feats['previous_invoice_count'] == 0, 'has_payment_history'] == 0).all()
assert ((hist_feats['previous_late_ratio'] >= 0) & (hist_feats['previous_late_ratio'] <= 1)).all()
# Counterfactual proof: first invoice per customer must have zero history.
firsts = paid_df.groupby('customer_id').head(1).index
assert (hist_feats.loc[firsts, 'previous_invoice_count'] == 0).all(), 'First invoice leaked history'
print('History OK. With history:', int((hist_feats['has_payment_history'] == 1).sum()), '/', len(hist_feats))
paid_hist = pd.concat([paid_df.reset_index(drop=True), hist_feats.reset_index(drop=True)], axis=1)
assert len(paid_hist) == len(paid_df)

## 10. Invoice-Level Feature Engineering

Feature-engine `DatetimeFeatures` (year/month/quarter/weekday/day-of-month/week/weekend) and `LogTransformer` for `log_invoice_amount`; `days_until_due = due_date - invoice_date` (uses only pre-payment dates, never `payment_date`); `cust_payment_terms` / `invoice_currency` retained as raw categoricals for train-fit encoding later.

In [ ]:
base = paid_hist.copy()
base['invoice_date'] = pd.to_datetime(base['invoice_date'], errors='coerce')
dt = DatetimeFeatures(variables=['invoice_date'], features_to_extract=['year', 'month', 'quarter', 'day_of_week', 'day_of_month', 'week', 'weekend'], drop_original=True).fit_transform(base[['invoice_date']].copy())
dt.columns = ['invoice_year', 'invoice_month', 'invoice_quarter', 'invoice_day_of_week', 'invoice_day_of_month', 'invoice_week_of_year', 'invoice_is_weekend']
base = pd.concat([base.reset_index(drop=True), dt.reset_index(drop=True)], axis=1)
base['days_until_due'] = (pd.to_datetime(base['due_date']) - pd.to_datetime(base['invoice_date'])).dt.days
lt = LogTransformer(variables=['invoice_amount'])
base['log_invoice_amount'] = lt.fit_transform(base[['invoice_amount']].copy())['invoice_amount']
assert np.isfinite(base['log_invoice_amount']).all(), 'Log transform produced non-finite values'
assert 'payment_date' not in ['invoice_amount', 'log_invoice_amount']
print(base[['invoice_amount', 'log_invoice_amount', 'invoice_year', 'invoice_month', 'days_until_due']].head(3).to_string(index=False))
print('terms/currency kept raw for train-fit encoding:', base['cust_payment_terms'].nunique(), 'terms,', base['invoice_currency'].nunique(), 'currencies')

## 11. Feature-engine Preprocessing Pipeline

Learned steps (fit on TRAIN only): `MeanMedianImputer(median)` for numerics; `RareLabelEncoder` + `OneHotEncoder` for `cust_payment_terms`/`invoice_currency`. History transformer is already applied (previous-only) and is NOT refit per split.

In [ ]:
NUMERIC_FEATURES = ['invoice_amount', 'log_invoice_amount', 'invoice_year', 'invoice_month', 'invoice_quarter',
                    'invoice_day_of_week', 'invoice_day_of_month', 'invoice_week_of_year', 'invoice_is_weekend',
                    'days_until_due', 'previous_invoice_count', 'previous_paid_invoice_count', 'previous_late_count',
                    'previous_late_ratio', 'previous_avg_delay', 'previous_median_delay', 'previous_max_delay',
                    'previous_min_delay', 'previous_avg_invoice_amount', 'previous_total_invoice_amount',
                    'previous_days_since_invoice', 'has_payment_history']
CATEGORICAL_FEATURES = ['cust_payment_terms', 'invoice_currency']
ID_COLS = ['invoice_id', 'customer_id', 'invoice_date', 'due_date']
assert TARGET not in NUMERIC_FEATURES + CATEGORICAL_FEATURES
preprocess = ColumnTransformer([
    ('num', MeanMedianImputer(imputation_method='median', variables=NUMERIC_FEATURES), NUMERIC_FEATURES),
    ('cat', Pipeline([('rare', RareLabelEncoder(tol=0.02, variables=CATEGORICAL_FEATURES)),
                       ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CATEGORICAL_FEATURES),
], verbose_feature_names_out=False)
print('numeric:', len(NUMERIC_FEATURES), '| categorical:', CATEGORICAL_FEATURES)

## 12. Chronological Train/Validation/Test Split

70/15/15 by row order on the chronologically sorted engineered table (no shuffling) to simulate future prediction. History was computed before the split, so validation/test keep valid earlier history and never future data.

In [ ]:
full = base.sort_values(['invoice_date', 'invoice_id'], kind='mergesort').reset_index(drop=True)
n = len(full)
n_train, n_val = int(n * 0.70), int(n * 0.15)
n_test = n - n_train - n_val
train_df = full.iloc[:n_train].copy()
val_df = full.iloc[n_train:n_train + n_val].copy()
test_df = full.iloc[n_train + n_val:].copy()
print(f'n={n} -> train {len(train_df)}, val {len(val_df)}, test {len(test_df)}')
for nm, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'{nm}: {d["invoice_date"].min().date()} -> {d["invoice_date"].max().date()}')
assert train_df['invoice_date'].max() <= val_df['invoice_date'].min()
assert val_df['invoice_date'].max() <= test_df['invoice_date'].min()
assert train_df['invoice_date'].is_monotonic_increasing and val_df['invoice_date'].is_monotonic_increasing

## 13. Fit Preprocessing ONLY on Training Data

Fit imputers/encoders on TRAIN exclusively.

In [ ]:
X_train_raw = train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_train = pd.to_numeric(train_df[TARGET], errors='coerce')
assert y_train.isna().sum() == 0
preprocess.fit(X_train_raw)
print('preprocessing fitted on train only:', X_train_raw.shape)
ohe = preprocess.named_transformers_['cat'].named_steps['ohe']
print('OHE categories:', [list(c)[:5] for c in ohe.categories_])

## 14. Transform Validation and Test Data

Transform val/test with the train-fitted pipeline; reattach identifiers + target. Assert no NaN/inf remains.

In [ ]:
def transformed_frame(df_part: pd.DataFrame) -> pd.DataFrame:
    Xt = preprocess.transform(df_part[NUMERIC_FEATURES + CATEGORICAL_FEATURES])
    feat_names = preprocess.get_feature_names_out().tolist()
    out = pd.DataFrame(Xt, columns=feat_names, index=df_part.index)
    assert np.isfinite(out.to_numpy(dtype=float)).all(), 'NaN/inf after preprocessing'
    for c in ID_COLS + [TARGET]:
        out[c] = df_part[c].values
    return out

train_t = transformed_frame(train_df)
val_t = transformed_frame(val_df)
test_t = transformed_frame(test_df)
print('transformed:', train_t.shape, val_t.shape, test_t.shape)
MODEL_FEATURES = [c for c in train_t.columns if c not in ID_COLS + [TARGET]]
print('model features:', len(MODEL_FEATURES))

## 15. Feature Validation

Target preserved signed; no forbidden column in X; no NaN/inf; history consistent.

In [ ]:
assert (train_t[TARGET] < 0).sum() > 0 and (test_t[TARGET] < 0).sum() > 0, 'Negatives lost'
leak_in_X = set(MODEL_FEATURES).intersection(FORBIDDEN)
print('forbidden in X:', leak_in_X if leak_in_X else 'none - PASS')
assert len(leak_in_X) == 0
for nm, d in [('train', train_t), ('val', val_t), ('test', test_t)]:
    assert np.isfinite(d[MODEL_FEATURES].to_numpy(dtype=float)).all(), f'{nm} has NaN/inf'
zero_hist = int((train_t['previous_invoice_count'] == 0).sum())
print(f'train rows without history: {zero_hist} ({100*zero_hist/len(train_t):.1f}%)')
print('target mean/median/std (train):', round(float(train_t[TARGET].mean()),3), round(float(train_t[TARGET].median()),3), round(float(train_t[TARGET].std()),3))

## 16. Leakage Audit

Audit table `Feature | Allowed? | Reason` covering target-derived, identifier and history features; saved to CSV.

In [ ]:
audit_rows = []
for f in FORBIDDEN:
    audit_rows.append({'Feature': f, 'Allowed?': 'NO',
        'Reason': 'Target-derived/post-payment/identifier - must never be a predictor' if f != 'customer_id' else 'Grouping key for history only, never a predictor'})
for f in NUMERIC_FEATURES:
    uses_hist = str(f.startswith('previous_') or f == 'has_payment_history')
    audit_rows.append({'Feature': f, 'Allowed?': 'YES',
        'Reason': f"Known at issue time; history uses strictly previous invoices only (uses_history={uses_hist})"})
for f in MODEL_FEATURES:
    if f not in NUMERIC_FEATURES and f not in FORBIDDEN:
        audit_rows.append({'Feature': f, 'Allowed?': 'YES', 'Reason': 'Train-fit one-hot of pre-payment categorical; unknown levels ignored'})
audit_df = pd.DataFrame(audit_rows)
print(audit_df.head(12).to_string(index=False))
print('forbidden allowed?', int((audit_df[audit_df['Feature'].isin(FORBIDDEN)]['Allowed?'] == 'YES').sum()), '(must be 0)')
assert (audit_df[audit_df['Feature'].isin(FORBIDDEN)]['Allowed?'] == 'YES').sum() == 0
AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
audit_df.to_csv(AUDIT_PATH, index=False)
print('saved:', AUDIT_PATH)

## 17. Feature Dictionary

`reports/payment_delay_feature_dictionary.csv` with `feature_name, feature_type, description, source_column, transformation, uses_customer_history, uses_target, leakage_safe, notes`.

In [ ]:
dict_rows = []
desc = {'invoice_amount': 'Invoice face amount (pre-payment)', 'log_invoice_amount': 'Natural-log invoice amount',
        'invoice_year': 'Calendar year of issue', 'invoice_month': 'Month of issue', 'invoice_quarter': 'Quarter of issue',
        'invoice_day_of_week': 'Weekday of issue', 'invoice_day_of_month': 'Day of month of issue',
        'invoice_week_of_year': 'ISO week of issue', 'invoice_is_weekend': 'Issue-date weekend flag',
        'days_until_due': 'Due_date minus invoice_date (payment terms length)',
        'previous_invoice_count': 'Count of strictly earlier invoices (customer)',
        'previous_paid_invoice_count': 'Count of strictly earlier paid invoices (paid-only frame)',
        'previous_late_count': 'Earlier invoices with delay_days>0', 'previous_late_ratio': 'Late share of earlier invoices (0 if none)',
        'previous_avg_delay': 'Mean delay_days of earlier invoices', 'previous_median_delay': 'Median delay of earlier invoices',
        'previous_max_delay': 'Max delay of earlier invoices', 'previous_min_delay': 'Min delay of earlier invoices',
        'previous_avg_invoice_amount': 'Mean amount of earlier invoices', 'previous_total_invoice_amount': 'Total amount of earlier invoices',
        'previous_days_since_invoice': 'Days since latest earlier invoice', 'has_payment_history': '1 if any earlier invoice exists'}
for f in NUMERIC_FEATURES:
    dict_rows.append({'feature_name': f, 'feature_type': 'numeric', 'description': desc.get(f, f),
        'source_column': 'invoice_amount/invoice_date/due_date' if not f.startswith('previous_') and f != 'has_payment_history' else 'customer history (previous invoices)',
        'transformation': ('LogTransformer' if f == 'log_invoice_amount' else ('raw amount' if f == 'invoice_amount' else ('DatetimeFeatures' if f.startswith('invoice_') else ('due-invoice days' if f == 'days_until_due' else 'previous-only aggregation + median imputation')))) ,
        'uses_customer_history': str(f.startswith('previous_') or f == 'has_payment_history'),
        'uses_target': 'YES (previous targets only)' if f in ['previous_late_count','previous_late_ratio','previous_avg_delay','previous_median_delay','previous_max_delay','previous_min_delay'] else 'NO',
        'leakage_safe': 'YES', 'notes': 'Neutral defaults when no history; current invoice never included'})
for f in MODEL_FEATURES:
    if f not in NUMERIC_FEATURES:
        dict_rows.append({'feature_name': f, 'feature_type': 'onehot', 'description': f'One-hot from {f}',
            'source_column': 'cust_payment_terms/invoice_currency', 'transformation': 'RareLabelEncoder(tol=0.02)+OneHotEncoder (fit train only)',
            'uses_customer_history': 'False', 'uses_target': 'NO', 'leakage_safe': 'YES', 'notes': 'Unknown categories ignored at transform'})
for f in FORBIDDEN:
    dict_rows.append({'feature_name': f, 'feature_type': 'excluded', 'description': 'Excluded from X',
        'source_column': f, 'transformation': 'none (target/grouping/identifier only)', 'uses_customer_history': 'n/a',
        'uses_target': 'YES' if f in ['delay_days','is_late'] else 'NO', 'leakage_safe': 'n/a (excluded)',
        'notes': 'Target' if f=='delay_days' else ('Grouping key only' if f=='customer_id' else 'Must never enter X')})
dict_df = pd.DataFrame(dict_rows)
dict_df.to_csv(DICT_PATH, index=False)
print(dict_df.head(8).to_string(index=False))
print('saved:', DICT_PATH, dict_df.shape)

## 18. Exploratory Feature Analysis

Distributions, delay over time, missing-history share and target relationships (descriptive only; no test-based selection). Plots saved to `reports/payment_delay/`.

In [ ]:
def savefig(name: str) -> None:
    p = PLOT_DIR / name
    plt.tight_layout(); plt.savefig(p, dpi=120); plt.show(); print('saved:', p)

plt.figure(figsize=(8,4)); plt.hist(train_t[TARGET], bins=60, edgecolor='black')
plt.title('Distribution of delay_days (train, signed)'); plt.xlabel('delay_days'); plt.ylabel('Invoices'); savefig('01_delay_distribution.png')
plt.figure(figsize=(8,3)); plt.boxplot(train_t[TARGET].clip(-60, 120), vert=False)
plt.title('Boxplot of delay_days (train, clipped for display)'); plt.xlabel('delay_days'); savefig('02_delay_boxplot.png')
plt.figure(figsize=(8,4)); plt.hist(train_t['invoice_amount'], bins=50, edgecolor='black')
plt.title('Invoice amount distribution (train)'); plt.xlabel('invoice_amount'); savefig('03_amount_distribution.png')
plt.figure(figsize=(8,4)); plt.hist(train_t['previous_avg_delay'].dropna(), bins=50, edgecolor='black')
plt.title('Previous average delay (train, has-history only)'); plt.xlabel('previous_avg_delay'); savefig('04_prev_avg_delay.png')
plt.figure(figsize=(8,4)); plt.hist(train_t['previous_late_ratio'], bins=20, edgecolor='black')
plt.title('Previous late ratio (train)'); plt.xlabel('previous_late_ratio'); savefig('05_prev_late_ratio.png')
monthly = train_df.assign(ym=pd.to_datetime(train_df['invoice_date']).dt.to_period('M').astype(str)).groupby('ym')[TARGET].mean()
plt.figure(figsize=(10,4)); plt.plot(monthly.index, monthly.values, marker='o')
plt.title('Mean delay_days over time (train)'); plt.xlabel('month'); plt.ylabel('mean delay'); plt.xticks(rotation=45); savefig('06_delay_over_time.png')
miss_pct = 100 * float((train_t['has_payment_history'] == 0).mean())
plt.figure(figsize=(5,4)); plt.bar(['with history','no history'], [100-miss_pct, miss_pct])
plt.title('History coverage (train)'); plt.ylabel('%'); savefig('07_history_coverage.png')
print(f'missing-history share (train): {miss_pct:.1f}%')
corr = train_t[MODEL_FEATURES + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET).sort_values(key=np.abs, ascending=False)
print('top |correlation| with delay_days (train):'); print(corr.head(10).round(3).to_string())
plt.figure(figsize=(8,5)); corr.head(12).iloc[::-1].plot(kind='barh')
plt.title('Correlation with delay_days (train, descriptive)'); plt.xlabel('correlation'); savefig('08_correlation.png')

## 19. Save Processed Datasets

Save full engineered features plus chronological train/val/test tables (transformed features + identifiers + signed target).

In [ ]:
FEATURES_PATH.parent.mkdir(parents=True, exist_ok=True)
full_save = full.copy()
full_save.to_csv(FEATURES_PATH, index=False)
train_t.to_csv(TRAIN_PATH, index=False)
val_t.to_csv(VAL_PATH, index=False)
test_t.to_csv(TEST_PATH, index=False)
print('saved:', FEATURES_PATH, full_save.shape)
print('saved:', TRAIN_PATH, train_t.shape)
print('saved:', VAL_PATH, val_t.shape)
print('saved:', TEST_PATH, test_t.shape)
DQ_PATH.parent.mkdir(parents=True, exist_ok=True)
dq_report.to_csv(DQ_PATH, index=False)
print('saved:', DQ_PATH)

## 20. Save Feature Pipeline

Persist the history transformer + train-fitted preprocessing + feature lists for reuse by the training notebook.

In [ ]:
PIPE_PATH.parent.mkdir(parents=True, exist_ok=True)
bundle = {'history_transformer': hist_tx, 'preprocess': preprocess,
          'numeric_features': NUMERIC_FEATURES, 'categorical_features': CATEGORICAL_FEATURES,
          'model_features': MODEL_FEATURES, 'id_cols': ID_COLS, 'target': TARGET,
          'train_date_range': [str(train_df['invoice_date'].min().date()), str(train_df['invoice_date'].max().date())],
          'val_date_range': [str(val_df['invoice_date'].min().date()), str(val_df['invoice_date'].max().date())],
          'test_date_range': [str(test_df['invoice_date'].min().date()), str(test_df['invoice_date'].max().date())]}
joblib.dump(bundle, PIPE_PATH)
print('saved:', PIPE_PATH)
reloaded = joblib.load(PIPE_PATH)
print('keys:', list(reloaded.keys()), '| model features:', len(reloaded['model_features']))

## 21. Final Summary

Usable invoices, features, splits, target stats, customers, history coverage and leakage result. Matrix reflects only information available at issue time.

In [ ]:
print('==================================================')
print('FLOWCRED AI - PAYMENT DELAY FEATURE ENGINEERING')
print('==================================================')
print(f'usable paid invoices: {len(full)}')
print(f'number of features: {len(MODEL_FEATURES)}')
print(f'training rows: {len(train_t)}')
print(f'validation rows: {len(val_t)}')
print(f'test rows: {len(test_t)}')
print(f"target mean/median/std: {train_t[TARGET].mean():.3f}/{train_t[TARGET].median():.3f}/{train_t[TARGET].std():.3f}")
print(f"min delay: {train_t[TARGET].min():.0f} | max delay: {train_t[TARGET].max():.0f}")
print(f"number of customers: {full['customer_id'].nunique()}")
print(f"with prior history (full paid frame): {100*float((hist_feats['has_payment_history']==1).mean()):.1f}%")
print('leakage audit: PASS (no forbidden column in X; history strictly previous-only)')
print('artifacts:', FEATURES_PATH.name, '|', TRAIN_PATH.name, '|', VAL_PATH.name, '|', TEST_PATH.name, '|', PIPE_PATH.name)
print('==================================================')
print('NEXT: regression training only (separate notebook). No model trained here.')